Historical reference (not the current scoring policy): # PP-OCRv6-derived price reader (custom)

This separate family is the measured price candidate. It uses a 128×320 BGR input, CTC decoding, reconstructed training-time BN, and a training-only PARSeq Tiny auxiliary reader. The auxiliary reader is discarded for inference. This is a local training recipe, not an official PP-OCRv6 model option.

Historical PP price result:161/206 literal exact. Historical timing/memory describe a specific checkpoint and environment; rebuilding needs fresh output and timing checks. No reference weights or private data are bundled.

Install `pip install -e ".[ppocr,onnx]"` for training/conversion and add `ppocr-tensorrt` for NVIDIA deployment. No pretrained/fine-tuned weights are bundled. Set dataset_root in configs/ppocr_price.yaml to Image/TXT train/val folders. Selection uses TRAIN only, then full-fit starts fresh from official initialization for the selected epoch count. External val never selects a checkpoint. Current selection is literal exact only, with earliest ties; numeric_then_exact is not supported. Legacy selection states require a new output directory.


Current training/evaluation policy: **literal-string exact only**, including spaces, symbols and case; earliest tied checkpoint wins. TXT permits one framing LF/CRLF only. Old training-selection state is incompatible with resume; historical reference cells retain their original scope. See [literal text policy](../docs/literal_text_policy.md).

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "parseq_ocr").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository")
sys.path.insert(0, str(ROOT))
from parseq_ocr.ppocr_config import load_config
CONFIG_PATH = os.environ.get('PPOCR_CONFIG', 'configs/ppocr_price.yaml')
cfg = load_config(CONFIG_PATH)
{k: str(v) for k, v in cfg.items()}

Run these steps only after checking your own paths. Official pretrained hashes are verified; local conversion keeps the original math before the explicit height/vocabulary adaptation. Fit-only images initialize BN statistics; holdout images never calibrate selection BN. Labels retain punctuation, spaces and cents and are never truncated. The current auxiliary decoder supports at most12 characters.


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    if cfg.get('recipe') == 'selection_only_precursor_v1':
        from parseq_ocr.ppocr_precursor import train
        result = train(cfg)
        checkpoint = Path(result['checkpoint'])
    elif cfg.get('recipe') == 'corrected_fixed_bn_v1':
        from parseq_ocr.ppocr_corrected_training import train
        result = train(cfg)
        checkpoint = Path(result['checkpoint'])
    else:
        from parseq_ocr.ppocr_training import train
        checkpoint = train(cfg)
    print(checkpoint)
else:
    print('Training disabled; edit config and opt in to run.')

In [ ]:
RUN_EXPORT = False
if RUN_EXPORT:
    from parseq_ocr.ppocr_export import export_onnx, build_engine
    checkpoint = cfg['output_dir'] / ('selection/model.pt' if cfg.get('recipe') == 'selection_only_precursor_v1' else 'full_fit/model.pt')
    deployment = export_onnx(checkpoint, cfg['output_dir'] / 'deployment')
    build_engine(deployment)

Equivalent CLI:

```bash
python -m parseq_ocr.ppocr_cli prepare --config configs/ppocr_price.yaml
python -m parseq_ocr.ppocr_cli train --config configs/ppocr_price.yaml
python -m parseq_ocr.ppocr_cli evaluate --config configs/ppocr_price.yaml
python -m parseq_ocr.ppocr_cli export --config configs/ppocr_price.yaml
python -m parseq_ocr.ppocr_cli build --config configs/ppocr_price.yaml
python -m parseq_ocr.ppocr_cli evaluate --config configs/ppocr_price.yaml --runtime tensorrt
```

Batch1 through32 is supported by the PP TensorRT profile. CTC collapses repeats before removing blank0; no country grammar or price substring extraction is applied. Report literal-string exact accuracy only; do not turn different currency/decimal/spacing strings into equivalent values. Benchmark image decoding consistently; predecoded RGB-to-text timing differs from file-to-text timing. WATERec orientation is a separate ALPR option and does not apply to this reader.


```bash
# File-to-text timing (batch1); batch32 reports amortized throughput.
python -m parseq_ocr.ppocr_cli benchmark --config configs/ppocr_price.yaml --runtime tensorrt --batch-size 1
python -m parseq_ocr.ppocr_cli benchmark --config configs/ppocr_price.yaml --runtime tensorrt --batch-size 32 --predecoded
```


## Reproduction scope
This notebook runs the public recipe with your data; it does not recreate the current corrected-reference experiment checkpoint. See [the reproduction boundary](../docs/experiments.md#reproduction-boundary-public-workflow-versus-measured-checkpoint). No measured weights/private data are bundled.


## Optional corrected-policy operations
Set `PPOCR_CONFIG=configs/ppocr_price_corrected.yaml` **before the setup cell**, and configure its required initial checkpoint + SHA256 first. The default remains the legacy preset. This option freezes source BN running statistics (affine trainable), uses effective16/micro1, no auxiliary reader,20epochs and epoch1–20 literal DEV selection; epoch0 is diagnostic only. `val` is therefore selection DEV, not an independent test.
The actual104 run inherited a locally selected proper-fitting BN+auxiliary warm start that is not bundled. This public option accepts your compatible hash-bound checkpoint; it never silently replaces it with generic weights. Public legacy training does not prove recreation of that historical initializer. See [the exact recipe and ancestry boundary](../docs/experiments.md#optional-corrected-price-pp-recipe).
Use `python -m parseq_ocr.ppocr_cli train --config configs/ppocr_price_corrected.yaml --smoke-steps 2` for a separate smoke folder before a full run. CPU smoke requires `device: cpu` and `precision: fp32`; it is not the measured CUDA BF16 operating point. No private weights/data or104-accuracy promise is included.


## Optional official-weight precursor → corrected recipe
This new recipe can start from the public official PP and PARSeqTiny downloads; it does not reproduce the private historical initializer or promise its accuracy. Use the same explicit ordered image_path/text FIT and DEV manifests for both stages. DEV is used for selection, never gradient updates or full refitting. Literal ASCII text (including spaces/case) and maximum auxiliary length 12 are enforced.
The precursor uses 30 epochs, B16, AdamW 1e-4 / wd 3e-5 / clip 5, FIT-only 128-image BN calibration in batches of32, then train-mode BN and auxiliary CTC+CE weight1. Workers default0. Corrected training then freezes BN and uses20 epochs/effective16/micro1. Historical labels, selection and conversion/environment differ; benchmark your own run.


In [ ]:
# Configure both JSONL paths in the precursor config before opting in.
RUN_PRECURSOR_CHAIN = False
if RUN_PRECURSOR_CHAIN:
    from parseq_ocr.ppocr_precursor import train as train_precursor
    from parseq_ocr.ppocr_corrected_training import train as train_corrected
    precursor_cfg = load_config("configs/ppocr_price_precursor.yaml")
    handoff = train_precursor(precursor_cfg)
    corrected_cfg = load_config("configs/ppocr_price_corrected.yaml")
    corrected_cfg.update(initial_checkpoint=handoff["checkpoint"],
        initial_checkpoint_sha256=handoff["checkpoint_sha256"],
        precursor_receipt=handoff["precursor_receipt"],
        train_manifest=precursor_cfg["train_manifest"],
        validation_manifest=precursor_cfg["validation_manifest"])
    train_corrected(corrected_cfg, smoke_steps=2)
    result = train_corrected(corrected_cfg)
    checkpoint = Path(result["checkpoint"])
    # Optional existing export/build workflow; GPU execution remains opt-in.
    # from parseq_ocr.ppocr_export import export_onnx, build_engine
    # deployment = export_onnx(checkpoint, corrected_cfg["output_dir"] / "deployment")
    # build_engine(deployment)
# A hash-bound user-supplied warm start remains available in the corrected config.
